In [10]:
import pandas as pd
from scipy.stats import wasserstein_distance
from scipy.stats import beta

In [11]:
scores = pd.read_pickle("hedging_word_scores.pkl")
scores

,hedging_word,all_scores,count,mean,std,alpha_param,beta_param
0,I am almost sure,"[0.7, 0.5, 0.7, 0.9, 0.3, 0.4, 0.6, 0.3, 0.3, ...",491,0.666415,0.198612,2.233129,1.081462
1,I am certain,"[0.9, 0.95, 0.95, 0.9, 0.95, 0.95, 1.0, 1.0, 0...",523,0.811205,0.271211,0.410787,0.207534
2,I am confident that,"[1.0, 1.0, 1.0, 0.9, 0.9, 0.9, 0.9, 0.9, 1.0, ...",522,0.703008,0.233378,0.910526,0.471482
3,I am convinced,"[1.0, 0.95, 0.95, 0.85, 0.9, 0.8, 0.8, 0.9, 0....",492,0.759146,0.252197,0.648560,0.385909
4,I am doubtful,"[0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2, ...",530,0.331830,0.178113,1.731108,3.489750
...,...,...,...,...,...,...,...
637,without doubt,"[0.7, 0.7, 0.7, 0.8, 0.8, 1.0, 1.0, 0.9, 0.95,...",500,0.896400,0.115668,3.665004,0.412903
638,without question,"[1.0, 1.0, 1.0, 1.0, 1.0, 0.0, 0.95, 0.0, 1.0,...",551,0.929782,0.110989,1.840769,0.222047
639,would,"[0.8, 0.7, 0.7, 0.7, 0.7, 0.3, 0.6, 0.6, 0.45,...",488,0.658934,0.187903,2.372313,1.213342
640,would appear to,"[0.45, 0.6, 0.5, 0.5, 0.3, 1.0, 0.6, 0.5, 0.6,...",594,0.578401,0.169973,3.472961,2.578178


In [12]:
def find_closest_hedging_words(target_alpha, target_beta, lexicon_df, top_k=5):
    """
    Find the top K closest hedging words to a target beta distribution.
    
    Args:
        target_alpha: alpha parameter of target beta distribution
        target_beta: beta parameter of target beta distribution
        lexicon_df: DataFrame with hedging words and their beta parameters
        top_k: number of closest matches to return
    
    Returns:
        DataFrame with top K closest hedging words and their distances
    """
    # Generate samples from target beta distribution
    target_samples = beta.rvs(target_alpha, target_beta, size=2000)
    
    distances = []
    
    for idx, row in lexicon_df.iterrows():
        if pd.isna(row['alpha_param']) or pd.isna(row['beta_param']):
            continue
        
        # Generate samples from hedging word's beta distribution
        word_samples = beta.rvs(row['alpha_param'], row['beta_param'], size=2000)
        
        # Compute Wasserstein distance
        w_dist = wasserstein_distance(target_samples, word_samples)
        
        distances.append({
            'hedging_word': row['hedging_word'],
            'alpha': row['alpha_param'],
            'beta': row['beta_param'],
            'mean': row['mean'],
            'wasserstein_distance': w_dist
        })
    
    # Sort by distance and return top K
    result_df = pd.DataFrame(distances).sort_values('wasserstein_distance').head(top_k)
    return result_df.reset_index(drop=True)

In [13]:
example_result = find_closest_hedging_words(target_alpha=1, target_beta=10, lexicon_df=scores, top_k=10)
example_result

,hedging_word,alpha,beta,mean,wasserstein_distance
0,with low confidence,0.322619,1.105849,0.282070,0.148558
1,I guess,1.062308,2.882891,0.281881,0.177978
2,I am unsure,0.779365,1.979616,0.309859,0.187418
3,I am uncertain,1.606339,3.937780,0.299567,0.195900
4,I am not sure,1.331371,2.698368,0.348585,0.233137
5,I am doubtful,1.731108,3.489750,0.331830,0.235575
6,I would guess,1.688285,3.271974,0.350527,0.257572
7,speculatively,1.605641,2.902793,0.358148,0.264861
8,it is questionable whether,3.460815,5.785430,0.374967,0.282646
9,it is doubtful that,1.592640,2.718630,0.381881,0.283034
